<a href="https://colab.research.google.com/github/PhucPower300121/FLUX-Jupyter/blob/main/flux_dev_i2i_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### FLUX.1-dev Image-to-Image — Colab (ComfyUI + GGUF Q4)

Credit: [ComfyUI](https://github.com/comfyanonymous/ComfyUI), [ComfyUI-GGUF (city96)](https://github.com/city96/ComfyUI-GGUF), [FLUX.1-dev-gguf (city96)](https://huggingface.co/city96/FLUX.1-dev-gguf)

In [ ]:
#@title Install ComfyUI + ComfyUI-GGUF + download model
%cd /content/
!git clone https://github.com/comfyanonymous/ComfyUI

%cd /content/ComfyUI

PIN_COMMIT = ""  # để trống = bản mới nhất, hoặc điền hash commit cụ thể để cố định version
if PIN_COMMIT:
    !git fetch --all -q
    !git reset --hard {PIN_COMMIT}

!pip install -q -r requirements.txt

# Cài custom node GGUF
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/city96/ComfyUI-GGUF comfyui_gguf
!pip install -q -r comfyui_gguf/requirements.txt

%cd /content/ComfyUI
import os
os.makedirs("/content/ComfyUI/models/unet", exist_ok=True)
os.makedirs("/content/ComfyUI/models/vae", exist_ok=True)
os.makedirs("/content/ComfyUI/models/clip", exist_ok=True)

!apt -y install -qq aria2

# GGUF Q4_K_S ~7GB, mmap load -> không đè RAM system trước khi qua GPU
unet_path = '/content/ComfyUI/models/unet/flux1-dev-Q4_K_S.gguf'
vae_path = '/content/ComfyUI/models/vae/ae.sft'
clip_l_path = '/content/ComfyUI/models/clip/clip_l.safetensors'
t5xxl_path = '/content/ComfyUI/models/clip/t5xxl_fp8_e4m3fn.safetensors'

if not os.path.exists(unet_path):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/flux1-dev-Q4_K_S.gguf -d /content/ComfyUI/models/unet -o flux1-dev-Q4_K_S.gguf
if not os.path.exists(vae_path):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/ae.sft -d /content/ComfyUI/models/vae -o ae.sft
if not os.path.exists(clip_l_path):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/clip_l.safetensors -d /content/ComfyUI/models/clip -o clip_l.safetensors
if not os.path.exists(t5xxl_path):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/t5xxl_fp8_e4m3fn.safetensors -d /content/ComfyUI/models/clip -o t5xxl_fp8_e4m3fn.safetensors

from IPython.display import clear_output
clear_output()

paths = [unet_path, vae_path, clip_l_path, t5xxl_path]
missing = [p for p in paths if not os.path.exists(p)]
if missing:
    print("\033[91mMISSING FILE (rerun this cell):\033[0m", missing)
else:
    print("\033[92mInsstall + download model complete.\033[0m")

In [ ]:
#@title Load model to RAM (GGUF, mmap -- low RAM)
%cd /content/ComfyUI

import random, torch, numpy as np
from PIL import Image
from nodes import NODE_CLASS_MAPPINGS
import nodes

# Import custom node GGUF đúng chuẩn package (nodes.py dùng relative import nội bộ)
import sys
sys.path.insert(0, "/content/ComfyUI/custom_nodes")
from importlib import import_module
gguf_pkg = import_module("comfyui_gguf")

UnetLoaderGGUF = gguf_pkg.NODE_CLASS_MAPPINGS["UnetLoaderGGUF"]()
DualCLIPLoader = NODE_CLASS_MAPPINGS["DualCLIPLoader"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
VAEEncode = NODE_CLASS_MAPPINGS["VAEEncode"]()
LoadImage = nodes.LoadImage()

with torch.inference_mode():
    clip = DualCLIPLoader.load_clip("t5xxl_fp8_e4m3fn.safetensors", "clip_l.safetensors", "flux")[0]
    unet = UnetLoaderGGUF.load_unet("flux1-dev-Q4_K_S.gguf")[0]
    vae = VAELoader.load_vae("ae.sft")[0]

print("Model load complete.")

In [ ]:
#@title Upload photos (Support to select multiple photos)
from google.colab import files
uploaded = files.upload()
import shutil, os
init_image_names = []
for fname in uploaded.keys():
    dst = os.path.join("/content/ComfyUI/input", fname)
    shutil.copy(fname, dst)
    init_image_names.append(fname)
print("Uploaded:", init_image_names)

In [ ]:
#@title Run Image-to-Image
positive_prompt = "for a hand caressing the character's head"  #@param {type:"string"}
negative_prompt = ""  #@param {type:"string"}
denoise = 0.5  #@param {type:"slider", min:0.1, max:1.0, step:0.05}
steps = 20  #@param {type:"slider", min:10, max:50, step:1}
cfg = 3.5  #@param {type:"number"}
sampler_name = "euler"  #@param ["euler", "euler_ancestral", "dpmpp_2m", "dpmpp_2m_sde"]
scheduler = "simple"  #@param ["simple", "normal", "karras"]
seed = 0  #@param {type:"integer"}

with torch.inference_mode():
    if seed == 0:
        seed = random.randint(0, 18446744073709551615)
    print("Seed:", seed)

    positive = CLIPTextEncode.encode(clip, positive_prompt)[0]
    negative = CLIPTextEncode.encode(clip, negative_prompt)[0]

    images = [LoadImage.load_image(fname)[0] for fname in init_image_names]
    latents = [VAEEncode.encode(vae, img)[0] for img in images]

    outputs = []
    for idx, latent_image in enumerate(latents):
        samples = KSampler.sample(
            unet, seed, steps, cfg, sampler_name, scheduler,
            positive, negative, latent_image, denoise=denoise
        )[0]
        decoded = VAEDecode.decode(vae, samples)[0].detach()
        out_img = Image.fromarray(np.array(decoded * 255, dtype=np.uint8)[0])
        out_path = f"/content/output_{idx}.png"
        out_img.save(out_path)
        outputs.append(out_path)
        print("Saved:", out_path)

outputs[-1] and Image.open(outputs[-1])

In [ ]:
#@title Download output
from google.colab import files
import glob
for p in sorted(glob.glob("/content/output_*.png")):
    files.download(p)